In [16]:
import os

print(os.listdir("/kaggle/input"))


['datasets']


In [17]:
import os

print(os.listdir("/kaggle/input/datasets"))

['briscdataset']


In [18]:
from pathlib import Path

for p in Path("/kaggle/input/datasets").iterdir():
    print(p)

/kaggle/input/datasets/briscdataset


In [19]:
from pathlib import Path

dataset_path = Path("/kaggle/input/datasets/briscdataset")

print("=== BRISC Dataset Structure ===")
for item in dataset_path.iterdir():
    print(f"📁 {item.name}")
    if item.is_dir():
        # Show subfolders (should be class names like glioma, meningioma, etc.)
        for subitem in list(item.iterdir())[:5]:  # Show first 5
            print(f"   └── {subitem.name}")
        if len(list(item.iterdir())) > 5:
            print(f"   ... and {len(list(item.iterdir())) - 5} more")

=== BRISC Dataset Structure ===
📁 brisc2025
   └── brisc2025


In [20]:
from pathlib import Path
from collections import Counter

dataset_path = Path("/kaggle/input/datasets/briscdataset")

# Find the training folder (could be 'training', 'train', etc.)
train_folder = None
for item in dataset_path.iterdir():
    if item.is_dir() and ('train' in item.name.lower() or 'training' in item.name.lower()):
        train_folder = item
        break

if train_folder is None:
    # If no train folder, use the first directory found
    train_folder = [p for p in dataset_path.iterdir() if p.is_dir()][0]

print(f"Using training folder: {train_folder}")

# Count images per class
class_counts = Counter()
for class_dir in train_folder.iterdir():
    if class_dir.is_dir():
        count = len(list(class_dir.glob("*.*")))
        class_counts[class_dir.name] = count
        print(f"{class_dir.name}: {count} images")

print(f"\nTotal images: {sum(class_counts.values())}")

Using training folder: /kaggle/input/datasets/briscdataset/brisc2025
brisc2025: 5 images

Total images: 5


In [21]:
from pathlib import Path

dataset_path = Path("/kaggle/input/datasets/briscdataset/brisc2025")

print("=== Full BRISC2025 Structure ===")
for item in dataset_path.iterdir():
    print(f"📁 {item.name} ({'dir' if item.is_dir() else 'file'})")
    if item.is_dir():
        for sub in item.iterdir():
            print(f"   └── {sub.name} ({'dir' if sub.is_dir() else 'file'})")
            if sub.is_dir():
                # Show class folders inside
                for cls in sub.iterdir():
                    print(f"       └── {cls.name}")

=== Full BRISC2025 Structure ===
📁 brisc2025 (dir)
   └── README.md (file)
   └── classification_task (dir)
       └── test
       └── train
   └── manifest.csv (file)
   └── manifest.csv.sha256 (file)
   └── manifest.json (file)
   └── manifest.json.sha256 (file)
   └── segmentation_task (dir)
       └── test
       └── train


In [22]:
from pathlib import Path
import pandas as pd

# Correct path with nested brisc2025 folder
dataset_path = Path("/kaggle/input/datasets/briscdataset/brisc2025/brisc2025")

print("=== Classification Task Structure ===")
cls_task = dataset_path / "classification_task"
for item in cls_task.iterdir():
    print(f"📁 {item.name}")
    if item.is_dir():
        for sub in item.iterdir():
            print(f"   └── {sub.name}")

# Check manifest
print("\n=== Manifest CSV ===")
manifest_path = dataset_path / "manifest.csv"
df = pd.read_csv(manifest_path)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\nFirst 5 rows:")
print(df.head())

=== Classification Task Structure ===
📁 test
   └── glioma
   └── meningioma
   └── no_tumor
   └── pituitary
📁 train
   └── glioma
   └── meningioma
   └── no_tumor
   └── pituitary

=== Manifest CSV ===
Shape: (15586, 16)
Columns: ['relative_path', 'filename', 'task', 'split', 'index', 'tumor_code', 'tumor_label', 'plane_code', 'plane_label', 'sequence', 'is_mask', 'linked_image', 'width', 'height', 'file_size_bytes', 'sha256']

First 5 rows:
                                       relative_path  \
0  classification_task\test\glioma\brisc2025_test...   
1  classification_task\test\glioma\brisc2025_test...   
2  classification_task\test\glioma\brisc2025_test...   
3  classification_task\test\glioma\brisc2025_test...   
4  classification_task\test\glioma\brisc2025_test...   

                            filename            task split  index tumor_code  \
0  brisc2025_test_00001_gl_ax_t1.jpg  classification  test      1         gl   
1  brisc2025_test_00002_gl_ax_t1.jpg  classification  

In [23]:
import pandas as pd
from pathlib import Path

dataset_path = Path("/kaggle/input/datasets/briscdataset/brisc2025/brisc2025")
manifest = pd.read_csv(dataset_path / "manifest.csv")

# Filter to classification task only, exclude masks
cls_manifest = manifest[
    (manifest['task'] == 'classification') & 
    (manifest['is_mask'] == False)
]

print("=== Official Splits ===")
print(cls_manifest['split'].value_counts())

print("\n=== Classes ===")
print(cls_manifest['tumor_label'].value_counts())

print("\n=== Cross-tab: split × class ===")
print(pd.crosstab(cls_manifest['split'], cls_manifest['tumor_label']))

=== Official Splits ===
split
train    5000
test     1000
Name: count, dtype: int64

=== Classes ===
tumor_label
pituitary     1757
meningioma    1635
glioma        1401
no_tumor      1207
Name: count, dtype: int64

=== Cross-tab: split × class ===
tumor_label  glioma  meningioma  no_tumor  pituitary
split                                               
test            254         306       140        300
train          1147        1329      1067       1457


In [24]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from tqdm import tqdm
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# ==========================================
# 1. CONFIGURATION
# ==========================================
DATASET_PATH = Path("/kaggle/input/datasets/briscdataset/brisc2025/brisc2025")
# CHANGE THIS to your actual Hugging Face username!
HF_REPO_ID = "srijan2715/brisc2025-paper-pipeline" 
SEED = 42

# ==========================================
# 2. LOAD MANIFEST & FILTER
# ==========================================
print("Loading manifest...")
manifest = pd.read_csv(DATASET_PATH / "manifest.csv")
cls_manifest = manifest[
    (manifest['task'] == 'classification') & 
    (manifest['is_mask'] == False)
].copy()

# Fix Windows paths for Linux
cls_manifest['relative_path'] = cls_manifest['relative_path'].str.replace('\\', '/')

# ==========================================
# 3. SPLIT DATA (Official Train -> Train/Val)
# ==========================================
train_df = cls_manifest[cls_manifest['split'] == 'train'].copy()
test_df = cls_manifest[cls_manifest['split'] == 'test'].copy()

# 85% train, 15% val for our experiments
train_split, val_split = train_test_split(
    train_df, test_size=0.15, 
    stratify=train_df['tumor_label'], 
    random_state=SEED
)
print(f"Splits created: Train={len(train_split)}, Val={len(val_split)}, Test={len(test_df)}")

# ==========================================
# 4. PREPROCESS (RESIZE TO 256x256)
# ==========================================
def process_images(df, split_name):
    images, labels = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name}"):
        img_path = DATASET_PATH / row['relative_path']
        try:
            img = Image.open(img_path).convert("RGB")
            # STRICT PAPER PIPELINE: Resize to 256 (CenterCrop 224 happens later in PyTorch)
            img_resized = img.resize((256, 256), Image.Resampling.BICUBIC)
            images.append(np.array(img_resized))
            labels.append(row['tumor_label'])
        except Exception as e:
            print(f"Failed: {img_path}")
    return images, labels

print("\nResizing images (this takes ~2 minutes)...")
train_imgs, train_lbls = process_images(train_split, "train")
val_imgs, val_lbls = process_images(val_split, "val")
test_imgs, test_lbls = process_images(test_df, "test")

# ==========================================
# 5. BUILD DATASET & UPLOAD
# ==========================================
dataset_dict = DatasetDict({
    "train": Dataset.from_dict({"image": train_imgs, "label": train_lbls}),
    "val": Dataset.from_dict({"image": val_imgs, "label": val_lbls}),
    "test": Dataset.from_dict({"image": test_imgs, "label": test_lbls})
})

print("\nAuthenticating with Hugging Face...")
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

print(f"Uploading to {HF_REPO_ID}...")
dataset_dict.push_to_hub(HF_REPO_ID, token=hf_token, private=True, max_shard_size="500MB")

print(f"\n🎉 DONE! Link: https://huggingface.co/datasets/{HF_REPO_ID}")

Loading manifest...
Splits created: Train=4250, Val=750, Test=1000

Resizing images (this takes ~2 minutes)...


Processing test: 100%|██████████| 1000/1000 [00:07<00:00, 125.62it/s]



Authenticating with Hugging Face...
Uploading to srijan2715/brisc2025-paper-pipeline...


Uploading the dataset shards:   0%|          | 0/4 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


🎉 DONE! Link: https://huggingface.co/datasets/srijan2715/brisc2025-paper-pipeline


In [25]:
# Create a notebooks directory
!mkdir -p notebooks

# 1. Create README.md
readme_content = """# Brain Tumor Classification: Reproducing Table 3 & DINOv2 Extension

## Objective
Faithfully reproduce Table 3 from the NeXtSwin-X paper (Esfahani et al., 2026) and extend the analysis with DINOv2 feature extraction and ablation studies.

## Datasets (Preprocessed)
To save compute and ensure faithful reproduction of the `timm` ImageNet pipeline (`Resize(256)` -> `CenterCrop(224)`), datasets are preprocessed and hosted on Hugging Face:
- **BRISC 2025**: [srijan2715/brisc2025-paper-pipeline](https://huggingface.co/datasets/srijan2715/brisc2025-paper-pipeline)
- **Sartaj et al.**: *(Coming soon)*

## Rules Followed
- [x] Git commit at every working session with meaningful messages.
- [x] All runs logged to Weights & Biases (W&B).
- [x] Lab notebook maintained with predictions, results, and failure analysis.
- [x] Strict adherence to paper's preprocessing and hyperparameter pipeline.
"""

with open("README.md", "w") as f:
    f.write(readme_content)

# 2. Create Lab Notebook Entry for Day 1
lab_notebook_content = """# Lab Notebook: Brain Tumor Classification Reproduction

## Session: Day 1 - Data Pipeline & Preprocessing
**Date:** 2024-10-24 (Adjust to actual date)
**Goal:** Faithfully reproduce the paper's preprocessing pipeline for Table 3.

### 1. What I Tried
- Explored the BRISC 2025 dataset structure and `manifest.csv`.
- Filtered for `task == 'classification'` and `is_mask == False` to match the paper's exact 5,000 train / 1,000 test split.
- Applied the strict `timm` ImageNet pipeline step 1: `Resize(256)` using `BICUBIC` interpolation. *(Note: `CenterCrop(224)` and `Normalize` are intentionally deferred to the PyTorch DataLoader to match `timm`'s exact behavior).*
- Split the official 5,000 train images into 85% Train (4,250) and 15% Val (750) using `random_state=42` for checkpoint selection.
- Uploaded processed tensors to Hugging Face Hub to eliminate CPU bottlenecking on Vast.ai.

### 2. What I Expected
- Expected exactly 5,000 train and 1,000 test images.
- Expected the resize operation to take ~2-3 minutes on Kaggle's free CPU.

### 3. What Happened (Results)
- **Manifest Check:** Confirmed exactly 5,000 train and 1,000 test images. Classes: glioma, meningioma, no_tumor, pituitary.
- **Preprocessing:** Successfully resized 6,000 images from 512x512 to 256x256.
- **Hugging Face Upload:** Successfully uploaded to `srijan2715/brisc2025-paper-pipeline`.

### 4. Analysis / Why
- Using the official manifest prevents data leakage, a common pitfall in custom random splits.
- Resizing to 256 (not 224) is critical. If we resize directly to 224, the subsequent `CenterCrop(224)` does nothing, altering the spatial distribution of the data and violating faithful reproduction.

### 5. Next Steps
- Repeat exact same pipeline for Sartaj et al. dataset.
- Phase 2: Hand-calculate ResNet50 parameters and implement from-scratch CNN.
"""

with open("notebooks/lab_notebook.md", "w") as f:
    f.write(lab_notebook_content)

print("✅ Files created successfully in /kaggle/working/")

✅ Files created successfully in /kaggle/working/


In [28]:
# Configure git identity (required for the first commit)
!git config --global user.name "Srijan27-lgtm"
!git config --global user.email "Srijan.verma274@gmail.com"

# Go to the working directory, add files, commit, and push
!cd /kaggle/working && \
git add . && \
git commit -m "feat: Day 1 - Dataset exploration, faithful preprocessing pipeline, and HF upload for BRISC 2025" && \
git push origin main

fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
